In [ ]:
import torch
import torch.nn as nn
from pathlib import Path

# ──────────────────────────────────────────────
# CONFIGURACIÓN GENERAL
# ──────────────────────────────────────────────
COMMANDS = ["yes", "no", "up", "down", "left", "right", "on", "off", "stop", "go"]
NUM_CLASSES = len(COMMANDS)
IMG_SIZE = 128
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ──────────────────────────────────────────────
# FUNCIÓN AUXILIAR: find_project_root
# ──────────────────────────────────────────────
def find_project_root(start: Path) -> Path:
    """Busca hacia arriba la carpeta que contiene `src` y `settings.gradle.kts`."""
    for p in [start, *start.parents]:
        if (p / "src").is_dir() and (p / "settings.gradle.kts").exists():
            return p
    raise FileNotFoundError("No se encontró la raíz del proyecto")

In [ ]:
# ──────────────────────────────────────────────
# CLASES AUXILIARES
# ──────────────────────────────────────────────
class ConvBNReLU(nn.Module):
    """Bloque básico: Conv2d + BatchNorm + ReLU6"""
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=0, groups=1):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, groups=groups, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu6 = nn.ReLU6(inplace=True)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu6(x)
        return x

class InvertedResidual(nn.Module):
    """Bloque Inverted Residual de MobileNetV2"""
    def __init__(self, in_channels, out_channels, stride, expand_ratio):
        super().__init__()
        self.stride = stride
        self.use_res_connect = (stride == 1 and in_channels == out_channels)

        hidden_dim = int(round(in_channels * expand_ratio))
        
        layers = []
        
        # Pointwise expansion (1x1)
        if expand_ratio != 1:
            layers.append(ConvBNReLU(in_channels, hidden_dim, kernel_size=1))
        
        # Depthwise (3x3 con groups=hidden_dim)
        layers.append(ConvBNReLU(hidden_dim, hidden_dim, kernel_size=3, stride=stride, padding=1, groups=hidden_dim))
        
        # Pointwise projection (1x1 lineal, sin activación)
        layers.append(nn.Conv2d(hidden_dim, out_channels, kernel_size=1, stride=1, padding=0, bias=False))
        layers.append(nn.BatchNorm2d(out_channels))
        
        self.conv = nn.Sequential(*layers)

    def forward(self, x):
        if self.use_res_connect:
            return x + self.conv(x)
        else:
            return self.conv(x)

In [ ]:
# ──────────────────────────────────────────────
# ARQUITECTURA: MobileNetV2Audio
# ──────────────────────────────────────────────
class MobileNetV2Audio(nn.Module):
    """
    MobileNetV2 construida completamente a mano para clasificación de audio.
    - Entrada: 1 canal (Mel-spectrogramas en escala de grises)
    - Arquitectura: Bloques Inverted Residual + Depthwise Separable Convolutions
    - Salida: NUM_CLASSES predicciones
    """
    def __init__(self, num_classes=10, width_mult=1.0):
        super().__init__()
        
        # Configuración de bloques (in_channels, out_channels, stride, expand_ratio, num_blocks)
        block_settings = [
            # t, c,  n, s
            (1,  16, 1, 1),
            (6,  24, 2, 2),
            (6,  32, 3, 2),
            (6,  64, 4, 2),
            (6,  96, 3, 1),
            (6, 160, 3, 2),
            (6, 320, 1, 1),
        ]
        
        in_channels = 32
        last_channels = 1280
        
        # Primera capa: Conv2d normal adaptada para 1 canal de entrada
        self.first_layer = ConvBNReLU(1, in_channels, kernel_size=3, stride=2, padding=1)
        
        # Bloques Inverted Residual
        features = []
        for t, c, n, s in block_settings:
            out_channels = int(c * width_mult)
            for i in range(n):
                stride = s if i == 0 else 1
                features.append(InvertedResidual(in_channels, out_channels, stride, t))
                in_channels = out_channels
        
        self.features = nn.Sequential(*features)
        
        # Última capa: Pointwise Conv (1x1)
        self.last_layer = ConvBNReLU(in_channels, last_channels, kernel_size=1)
        
        # Classifier: GlobalAveragePooling + Dropout + Linear
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(p=0.2)
        self.classifier = nn.Linear(last_channels, num_classes)
        
        # Inicializar pesos
        self._init_weights()

    def _init_weights(self):
        """Inicialización de pesos según He/Xavier"""
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.first_layer(x)
        x = self.features(x)
        x = self.last_layer(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.classifier(x)
        return x

In [ ]:
# ──────────────────────────────────────────────
# EXPORTACIÓN A ONNX — MODELO B, CONFIG-3
# ──────────────────────────────────────────────
# Ruta del modelo guardado por Config-3 (Modelo B sin datos aumentados)
base_dir = find_project_root(Path.cwd()) / "modelos"
best_model_path = base_dir / "best_model_b_raw_config3.pth"
onnx_output_path = base_dir / "modelo_b_config3.onnx"

# Instanciar arquitectura e inicializar pesos del Config-3
model_b_config3 = MobileNetV2Audio(num_classes=NUM_CLASSES)
model_b_config3.load_state_dict(torch.load(best_model_path, map_location="cpu"))
model_b_config3.eval()  # Modo inferencia (desactiva Dropout y BatchNorm en modo train)

# Tensor de ejemplo que coincide con el input real del modelo:
# batch=1, canales=1 (escala de grises), alto=128, ancho=128
input_tensor = torch.rand((1, 1, IMG_SIZE, IMG_SIZE), dtype=torch.float32)

# Exportar a ONNX
torch.onnx.export(
    model_b_config3,            # Modelo a exportar
    (input_tensor,),            # Entrada de ejemplo
    str(onnx_output_path),      # Ruta de salida
    input_names=["input"],      # Nombre del tensor de entrada
    output_names=["logits"],    # Nombre del tensor de salida
    dynamic_axes={              # Ejes dinámicos (batch size variable)
        "input":  {0: "batch_size"},
        "logits": {0: "batch_size"},
    },
    dynamo=False,               # False = exportador TorchScript (más estable)
    opset_version=17,           # Opset compatible con Android/iOS
)

print(f" Modelo exportado a: {onnx_output_path}")

# ── Verificación opcional con onnx ──────────────────────────────
try:
    import onnx
    model_onnx = onnx.load(str(onnx_output_path))
    onnx.checker.check_model(model_onnx)
    print(" Verificación ONNX: modelo válido")
except ImportError:
    print("  Instala 'onnx' para verificar: pip install onnx")